# 🤝 SF Community Events (Our415) - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

[Our415](https://www.our415.org/) is the city's directory of free and low-cost **events and activities**, mostly for kids, teens, and families: library story hours in a dozen languages, rec center drop-ins, sports leagues, art classes, job programs. This notebook loads the current listings into Elasticsearch.

**What this notebook does:**
1. Fetches the listings from [DataSF](https://data.sf.gov/d/8i3s-ih2a) - no API key required
2. Normalises dates, age groups, fee, neighborhood, and builds a `geo_point`
3. Bulk-indexes everything into `sf_community_events`
4. **(Optional)** translates event descriptions with **Mistral**

---
> **Data source:** Our415 Events and Activities, dataset `8i3s-ih2a`. Refreshed daily from SF Rec & Park, SF Public Library, and community organisations.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import re, requests
from datetime import date
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch the listings
A couple of thousand rows - we take them all.

In [ ]:
resp = requests.get('https://data.sf.gov/resource/8i3s-ih2a.json', params={'$limit': 50000}, timeout=60)
resp.raise_for_status()
rows = resp.json()
print(f'✅ Fetched {len(rows)} events and activities')
print()
for r in rows[:3]:
    print(f" • [{r.get('org_name')}] {r.get('event_name')} - {r.get('site_location_name')}")

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'sf_community_events'

mapping = {'mappings': {'properties': {
    'event_id':     {'type': 'keyword'},
    'organization': {'type': 'keyword'},   # SF Rec Park, SF Public Library, ...
    'name':         {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'description':  {'type': 'text'},
    'category':     {'type': 'keyword'},
    'age_groups':   {'type': 'keyword'},   # Babies, Toddlers, Children, Pre-Teens, Teens, ...
    'start_date':   {'type': 'date'},
    'end_date':     {'type': 'date'},
    'days_of_week': {'type': 'keyword'},
    'start_time':   {'type': 'keyword'},
    'end_time':     {'type': 'keyword'},
    'is_free':      {'type': 'boolean'},
    'more_info':    {'type': 'keyword'},
    'venue':        {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'address':      {'type': 'text'},
    'neighborhood': {'type': 'keyword'},
    'location':     {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest
Age groups arrive as free text in two different vocabularies (`Children; Pre-Teens` from Rec & Park, `Children/Pre-Teens (Ages 5-13)` from the library); we map both onto one list of groups so you can filter on a single one. `fee` is a "has a fee" flag, so we flip it into `is_free`.

In [ ]:
def geo_point(lat, lon):
    """Build a geo_point from lat/lon strings, skipping blanks and 0,0."""
    try:
        lat_f, lon_f = float(lat), float(lon)
    except (TypeError, ValueError):
        return None
    if lat_f == 0.0 or lon_f == 0.0:
        return None
    return {'lat': lat_f, 'lon': lon_f}

AGE_GROUPS = [   # (group, pattern found in the raw tags)
    ('Babies',             r'Bab|Infant'),
    ('Toddlers',           r'Toddler'),
    ('Children',           r'Children'),
    ('Pre-Teens',          r'Pre-Teen'),
    ('Teens',              r'(?<!Pre-)Teens'),
    ('Young Adults 18-24', r'TAY'),
    ('Families',           r'Famil'),
    ('All Ages',           r'All Ages'),
]

def age_groups(tags):
    return [g for g, pattern in AGE_GROUPS if re.search(pattern, tags or '')]

def enrich(r):
    fee = r.get('fee')
    doc = {
        'event_id':     r.get('id'),
        'organization': r.get('org_name'),
        'name':         r.get('event_name'),
        'description':  ' '.join((r.get('event_description') or '').split()),
        'category':     r.get('events_category'),
        'age_groups':   age_groups(r.get('age_group_eligibility_tags')),
        'start_date':   r.get('event_start_date'),
        'end_date':     r.get('event_end_date'),
        'days_of_week': (r.get('days_of_week') or '').split(),
        'start_time':   r.get('start_time'),
        'end_time':     r.get('end_time'),
        'is_free':      (not fee) if isinstance(fee, bool) else None,
        'more_info':    r.get('more_info'),
        'venue':        r.get('site_location_name'),
        'address':      r.get('site_address'),
        'neighborhood': r.get('analysis_neighborhood'),
        'location':     geo_point(r.get('latitude'), r.get('longitude')),
    }
    return {k: v for k, v in doc.items() if v not in (None, '', [])}

docs = [enrich(r) for r in rows]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 5. Verify - what's on?

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'aggs': {
    'by_category': {'terms': {'field': 'category', 'size': 10}},
    'by_org':      {'terms': {'field': 'organization', 'size': 10}},
    'by_age':      {'terms': {'field': 'age_groups', 'size': 10}},
}})
print(f"Total listings: {resp['hits']['total']['value']}\n")
for agg, title in [('by_category', 'Categories'), ('by_org', 'Organizations'), ('by_age', 'Age groups')]:
    print(f'{title}:')
    for b in resp['aggregations'][agg]['buckets']:
        print(f"  {b['key'][:44]:44s} {b['doc_count']}")
    print()

In [ ]:
# Free events running today, nearest to the venue first
today = date.today().isoformat()
resp = es.search(index=INDEX, body={'size': 8,
    'query': {'bool': {'filter': [
        {'term': {'is_free': True}},
        {'range': {'start_date': {'lte': today}}},
        {'range': {'end_date': {'gte': today}}},
        {'exists': {'field': 'location'}},
    ]}},
    'sort': [{'_geo_distance': {'location': {'lat': 37.7873, 'lon': -122.4005}, 'order': 'asc', 'unit': 'km'}}],
    'collapse': {'field': 'name.keyword'},   # one row per activity, not per session
    '_source': ['name', 'venue', 'neighborhood', 'start_time'],
})
print(f"Free events running today: {resp['hits']['total']['value']}")
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {h['sort'][0]:4.1f} km  {s['name'][:45]:45s} {s.get('venue', '')} ({s.get('neighborhood', '')})")

## 6. (Optional) Translate a listing with Mistral 🌐
About a third of San Franciscans speak a language other than English at home. Translate a few listings into Spanish, Chinese, Tagalog, Vietnamese, or Russian.

> Needs your **Mistral API key**.

In [ ]:
MISTRAL_API_KEY = '<MISTRAL_API_KEY>'  # 🔑 paste your Mistral key
LANGUAGE = 'Spanish'

def translate(text):
    r = requests.post('https://api.mistral.ai/v1/chat/completions',
        headers={'Authorization': f'Bearer {MISTRAL_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'mistral-small-latest', 'messages': [
            {'role': 'system', 'content': f'Translate the user text into {LANGUAGE}. Reply with the translation only.'},
            {'role': 'user', 'content': text}]},
        timeout=60)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

for h in es.search(index=INDEX, body={'size': 3, 'query': {'match': {'category': 'Sports & Recreation'}}})['hits']['hits']:
    s = h['_source']
    print(f"{s['name']}\n  → {translate(s['name'] + '. ' + s.get('description', ''))}\n")

## 7. Going further with Mistral

- **"What can my kid do this Saturday?"** A Mistral agent with an Elasticsearch search tool that filters by age group, day, price, and distance, and answers in the parent's language.
- **Semantic search:** back `description` with Mistral embeddings so "something creative for a shy 8-year-old" finds the art classes. See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).
- **Voice:** a phone-style assistant that takes a spoken question with **Voxtral**, searches here, and reads back the answer.
- **Combine:** with [sf_parks_and_public_art](sf_parks_and_public_art.ipynb) and [muni_schedule](muni_schedule.ipynb) to say how to get there.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`sf_community_events` is ready. Try "free teen events in the Excelsior" or "story hours in Cantonese".

**To refresh the data**, re-run cells 2-4. The index is dropped and recreated each time, so there are no duplicates.